# 1. Veri Yükleme ve Keşifsel Analiz

Bu notebook'ta yıldırım ve meteorolojik verileri yükleyip inceleyeceğiz.

**İçerik:**
- Kütüphanelerin yüklenmesi
- Yıldırım verisinin okunması
- Meteorolojik verilerin okunması
- Temel istatistikler
- Görselleştirmeler

In [ ]:
# Gerekli Kütüphaneler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import os
from scipy.spatial.distance import cdist

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Türkçe karakter desteği
plt.rcParams['font.family'] = 'DejaVu Sans'

print('Kütüphaneler yüklendi!')

## 1.1 Veri Yolu Tanımları

In [ ]:
# Veri klasörü yolu
DATA_PATH = '../data/'

# Dosya listesi
dosyalar = os.listdir(DATA_PATH)
print("Veri klasöründeki dosyalar:")
for d in dosyalar:
    print(f"  - {d}")

## 1.2 İstasyon Bilgileri

In [ ]:
# İstasyon bilgileri
istasyonlar = pd.DataFrame({
    'istasyon_no': [17285, 17920, 18234, 18771, 17815, 19909],
    'istasyon_adi': ['HAKKARİ', 'YÜKSEKOVA', 'ŞEMDİNLİ', 'HAKKARİ/DURANKAYA', 
                    'YÜKSEKOVA HAVALİMANI', 'HAKKARİ KAYAK MERKEZİ'],
    'il': ['Hakkari'] * 6,
    'ilce': ['Merkez', 'Yüksekova', 'Şemdinli', 'Merkez', 'Yüksekova', 'Merkez'],
    'enlem': [37.5745, 37.5785, 37.2950, 37.56, 37.5481, 37.571393],
    'boylam': [43.7388, 44.2862, 44.5819, 43.6094, 44.2399, 43.6818],
    'rakim': [1727, 1877, 1284, 1825, 1854, 2516]
})

print("İstasyon Bilgileri:")
display(istasyonlar)

## 1.3 Yıldırım Verisinin Yüklenmesi

In [ ]:
# Yıldırım verisi dosya adı
yildirim_dosya = [f for f in dosyalar if 'Yıldırım' in f or 'Y?ld?r?m' in f][0]
print(f"Yıldırım dosyası: {yildirim_dosya}")

# Veriyi oku
yildirim_df = pd.read_csv(
    DATA_PATH + yildirim_dosya,
    sep='|',
    encoding='latin-1',
    header=0
)

# Sütun isimlerini düzenle
yildirim_df.columns = ['zaman', 'enlem', 'boylam', 'yukseklik_km', 'akim_kA', 'olay_tipi', 'mesafe_km']

print(f"\nToplam kayıt sayısı: {len(yildirim_df):,}")
print(f"\nİlk 5 satır:")
display(yildirim_df.head())

In [ ]:
# Zaman sütununu datetime'a çevir
yildirim_df['zaman'] = pd.to_datetime(yildirim_df['zaman'], format='%Y-%m-%d %H:%M:%S.%f')
yildirim_df['tarih'] = yildirim_df['zaman'].dt.date
yildirim_df['yil'] = yildirim_df['zaman'].dt.year
yildirim_df['ay'] = yildirim_df['zaman'].dt.month
yildirim_df['gun'] = yildirim_df['zaman'].dt.day
yildirim_df['saat'] = yildirim_df['zaman'].dt.hour

print("Veri tipi dönüşümü tamamlandı!")
print(f"\nTarih aralığı: {yildirim_df['tarih'].min()} - {yildirim_df['tarih'].max()}")

In [ ]:
# Olay tipi dağılımı
print("Olay Tipi Dağılımı:")
print(yildirim_df['olay_tipi'].value_counts())

In [ ]:
# Temel istatistikler
print("Sayısal Değişkenler için İstatistikler:")
display(yildirim_df[['enlem', 'boylam', 'yukseklik_km', 'akim_kA', 'mesafe_km']].describe())

## 1.4 Yıldırım Verisi Görselleştirmeleri

In [ ]:
# Yıllık yıldırım sayısı
fig, ax = plt.subplots(figsize=(12, 5))
yillik_sayi = yildirim_df.groupby('yil').size()
yillik_sayi.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_xlabel('Yıl')
ax.set_ylabel('Yıldırım/Şimşek Sayısı')
ax.set_title('Yıllara Göre Yıldırım/Şimşek Olayları')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../tez_docs/figures/yillik_yildirim.png', dpi=150)
plt.show()

In [ ]:
# Aylık dağılım
fig, ax = plt.subplots(figsize=(10, 5))
aylik_sayi = yildirim_df.groupby('ay').size()
ay_isimleri = ['Oca', 'Şub', 'Mar', 'Nis', 'May', 'Haz', 'Tem', 'Ağu', 'Eyl', 'Eki', 'Kas', 'Ara']
aylik_sayi.index = ay_isimleri
aylik_sayi.plot(kind='bar', ax=ax, color='coral', edgecolor='black')
ax.set_xlabel('Ay')
ax.set_ylabel('Yıldırım/Şimşek Sayısı')
ax.set_title('Aylara Göre Yıldırım/Şimşek Dağılımı')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../tez_docs/figures/aylik_yildirim.png', dpi=150)
plt.show()

In [ ]:
# Saatlik dağılım
fig, ax = plt.subplots(figsize=(12, 5))
saatlik_sayi = yildirim_df.groupby('saat').size()
saatlik_sayi.plot(kind='bar', ax=ax, color='seagreen', edgecolor='black')
ax.set_xlabel('Saat')
ax.set_ylabel('Yıldırım/Şimşek Sayısı')
ax.set_title('Saatlere Göre Yıldırım/Şimşek Dağılımı')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../tez_docs/figures/saatlik_yildirim.png', dpi=150)
plt.show()

In [ ]:
# Coğrafi dağılım
fig, ax = plt.subplots(figsize=(10, 8))

# Yıldırım noktaları (örnek - tümünü çizmek yavaş olabilir)
sample = yildirim_df.sample(min(10000, len(yildirim_df)))
ax.scatter(sample['boylam'], sample['enlem'], alpha=0.3, s=1, c='blue', label='Yıldırım/Şimşek')

# İstasyon konumları
ax.scatter(istasyonlar['boylam'], istasyonlar['enlem'], c='red', s=100, marker='^', 
           edgecolors='black', label='Meteoroloji İstasyonu', zorder=5)

for idx, row in istasyonlar.iterrows():
    ax.annotate(row['istasyon_adi'], (row['boylam'], row['enlem']), 
                xytext=(5, 5), textcoords='offset points', fontsize=8)

ax.set_xlabel('Boylam')
ax.set_ylabel('Enlem')
ax.set_title('Yıldırım/Şimşek ve Meteoroloji İstasyonları Konumları')
ax.legend()
plt.tight_layout()
plt.savefig('../tez_docs/figures/cografi_dagilim.png', dpi=150)
plt.show()

## 1.5 Meteorolojik Verilerin Yüklenmesi

In [ ]:
def yukle_meteoroloji(dosya_adi, veri_adi):
    """Meteorolojik Excel dosyasını yükler ve düzenler."""
    try:
        df = pd.read_excel(DATA_PATH + dosya_adi)
        print(f"{veri_adi}: {df.shape[0]} satır, {df.shape[1]} sütun")
        return df
    except Exception as e:
        print(f"Hata ({veri_adi}): {e}")
        return None

# Meteorolojik verileri yükle
meteo_dosyalar = {
    'maks_sicaklik': [f for f in dosyalar if 'Maksimum Sıcaklık' in f or 'Maksimum S?cakl?k' in f][0],
    'min_sicaklik': [f for f in dosyalar if 'Minimum Sıcaklık' in f or 'Minimum S?cakl?k' in f][0],
    'ort_sicaklik': [f for f in dosyalar if 'Ortalama Sıcaklık' in f or 'Ortalama S?cakl?k' in f][0],
    'basinc': [f for f in dosyalar if 'Basınç' in f or 'Bas?n?' in f][0],
    'nem': [f for f in dosyalar if 'Nem' in f][0],
    'bulut': [f for f in dosyalar if 'Bulutluluk' in f][0],
    'ruzgar': [f for f in dosyalar if 'Rüzgar' in f or 'R?zgar' in f][0],
}

# Yağış dosyaları
yagis_dosyalari = [f for f in dosyalar if 'Yağış' in f or 'Ya???' in f]
if yagis_dosyalari:
    meteo_dosyalar['yagis'] = yagis_dosyalari[0]

print("Yüklenecek dosyalar:")
for k, v in meteo_dosyalar.items():
    print(f"  {k}: {v}")

In [ ]:
# Tüm meteorolojik verileri yükle
meteo_veriler = {}

for veri_adi, dosya in meteo_dosyalar.items():
    meteo_veriler[veri_adi] = yukle_meteoroloji(dosya, veri_adi)

In [ ]:
# Örnek: Maksimum sıcaklık verisinin ilk satırları
if meteo_veriler['maks_sicaklik'] is not None:
    print("\nMaksimum Sıcaklık Verisi (İlk 5 satır):")
    display(meteo_veriler['maks_sicaklik'].head())

In [ ]:
# Eksik veri analizi
print("\nEksik Veri Analizi:")
for veri_adi, df in meteo_veriler.items():
    if df is not None:
        eksik_oran = (df.isnull().sum().sum() / df.size) * 100
        print(f"  {veri_adi}: %{eksik_oran:.2f} eksik")

## 1.6 Günlük Yıldırım Sayıları

In [ ]:
# Günlük yıldırım sayısı
gunluk_yildirim = yildirim_df.groupby('tarih').size().reset_index(name='yildirim_sayisi')
gunluk_yildirim['tarih'] = pd.to_datetime(gunluk_yildirim['tarih'])

print(f"Yıldırım olan gün sayısı: {len(gunluk_yildirim)}")
print(f"\nGünlük yıldırım sayısı istatistikleri:")
display(gunluk_yildirim['yildirim_sayisi'].describe())

In [ ]:
# Günlük yıldırım sayısı dağılımı
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(gunluk_yildirim['yildirim_sayisi'], bins=50, color='purple', edgecolor='black', alpha=0.7)
ax.set_xlabel('Günlük Yıldırım Sayısı')
ax.set_ylabel('Frekans')
ax.set_title('Günlük Yıldırım Sayısı Dağılımı')
plt.tight_layout()
plt.savefig('../tez_docs/figures/gunluk_yildirim_dagilim.png', dpi=150)
plt.show()

## 1.7 Verilerin Kaydedilmesi

In [ ]:
# Ara verileri kaydet
yildirim_df.to_pickle('../data/yildirim_processed.pkl')
gunluk_yildirim.to_pickle('../data/gunluk_yildirim.pkl')
istasyonlar.to_pickle('../data/istasyonlar.pkl')

for veri_adi, df in meteo_veriler.items():
    if df is not None:
        df.to_pickle(f'../data/meteo_{veri_adi}.pkl')

print("Veriler kaydedildi!")

---
**Sonraki Adım:** `02_veri_onisleme.ipynb` - Veri Birleştirme ve Temizleme